# Tahap 5: Data Transformation & Discretization
**Silabus:** Jiawei Han - Data Mining: Concepts and Techniques, Ch. 3 (Data Transformation and Discretization)  
**DataFrame Target:** `df_clean` (Data Hasil Preprocessing Step 2, 3, & 4)  
**Fokus:**  
1. **Data Discretization & Binning Analysis**: Pembanding Equal-Width vs Equal-Frequency Partitioning pada atribut skewed `lead_time`, Visualisasi Histogram/Bar Chart Side-by-Side, dan Penerapan `lead_time_category`.  
2. **Concept Hierarchy Generation**: Pembentukan `arrival_date` (datetime), `arrival_quarter`, dan `arrival_season` untuk abstraksi hierarki dimensi waktu (OLAP / Data Cube Roll-up).  
3. **Penyimpanan Data Final**: Simpan data hasil olahan ke `hotel_booking_preprocessed.csv`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# 0. Memuat & Menyiapkan df_clean (Step 2, 3, 4 Preprocessing)
csv_file = "hotel_booking_data.csv"
df_raw = pd.read_csv(csv_file)

cols_to_drop = ['name', 'email', 'phone-number', 'credit_card', 'reservation_status', 'reservation_status_date']
df_clean = df_raw.drop(columns=[c for c in cols_to_drop if c in df_raw.columns]).drop_duplicates()

adr_anom = (df_clean['adr'] < 0) | (df_clean['adr'] > 1000)
total_guests = df_clean['adults'] + df_clean['children'].fillna(0) + df_clean['babies'].fillna(0)
df_clean = df_clean[~adr_anom & (total_guests > 0)].copy()

df_clean['children'] = df_clean['children'].fillna(0).astype(int)
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['agent'] = df_clean['agent'].fillna(0)
df_clean['company'] = df_clean['company'].fillna(0)

# Step 4 Feature Extraction
df_clean['total_stay_nights'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']
df_clean['is_room_changed'] = (df_clean['reserved_room_type'] != df_clean['assigned_room_type']).astype(int)

# =========================================================
# 1. DATA DISCRETIZATION & BINNING ANALYSIS (SLIDE 26-28)
# =========================================================
print("=" * 80)
print("1. DATA DISCRETIZATION & BINNING ANALYSIS: ATRIBUT lead_time")
print("=" * 80)

# Metode 1: Equal-Width Partitioning (N = 4)
df_clean['lead_time_eq_width'] = pd.cut(df_clean['lead_time'], bins=4)

# Metode 2: Equal-Frequency / Equal-Depth Partitioning (N = 4)
labels = ['Very Short', 'Short', 'Medium', 'Long']
df_clean['lead_time_category'] = pd.qcut(df_clean['lead_time'], q=4, labels=labels)

# Hitung Frekuensi
eq_width_counts = df_clean['lead_time_eq_width'].value_counts().sort_index()
eq_freq_counts = df_clean['lead_time_category'].value_counts()[labels]

print("\n--- Frekuensi Binning Equal-Width Partitioning ---")
display(pd.DataFrame({'Interval Equal-Width': eq_width_counts.index.astype(str), 'Jumlah Record': eq_width_counts.values}))

print("\n--- Frekuensi Binning Equal-Frequency Partitioning ---")
display(pd.DataFrame({'Kategori Equal-Frequency': eq_freq_counts.index, 'Jumlah Record': eq_freq_counts.values}))

# Visualisasi Side-by-Side (Bar Chart)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Equal-Width
axes[0].bar(range(4), eq_width_counts.values, color='#e74c3c', alpha=0.85)
axes[0].set_xticks(range(4))
axes[0].set_xticklabels([f"Bin {i+1}" for i in range(4)])
axes[0].set_title("Equal-Width Partitioning (Miring ke Bin 1)", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Jumlah Record (Frekuensi)", fontweight='bold')
for i, v in enumerate(eq_width_counts.values):
    axes[0].text(i, v + 1000, f"{v:,}", ha='center', fontweight='bold', fontsize=9)

# Plot 2: Equal-Frequency
axes[1].bar(labels, eq_freq_counts.values, color='#2ecc71', alpha=0.85)
axes[1].set_title("Equal-Frequency Partitioning (Masing-masing ~25%)", fontsize=12, fontweight='bold')
axes[1].set_ylabel("Jumlah Record (Frekuensi)", fontweight='bold')
for i, v in enumerate(eq_freq_counts.values):
    axes[1].text(i, v + 500, f"{v:,}", ha='center', fontweight='bold', fontsize=9)

plt.suptitle("Perbandingan Metode Diskretisasi pada Atribut Right-Skewed 'lead_time'", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Bersihkan kolom temporary
df_clean.drop(columns=['lead_time_eq_width'], inplace=True)


# =========================================================
# 2. CONCEPT HIERARCHY GENERATION (SLIDE 30-32)
# =========================================================
print("\n" + "=" * 80)
print("2. CONCEPT HIERARCHY GENERATION (ABSTRAKSI DIMENSI WAKTU)")
print("=" * 80)

# Konversi bulan string ke angka
month_map = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
    'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
}
df_clean['month_num'] = df_clean['arrival_date_month'].map(month_map)

# Datetime tunggal
df_clean['arrival_date'] = pd.to_datetime(
    df_clean['arrival_date_year'].astype(str) + '-'
    + df_clean['month_num'].astype(str).str.zfill(2) + '-'
    + df_clean['arrival_date_day_of_month'].astype(str).str.zfill(2)
)

# Hierarki Kuartal (Q1, Q2, Q3, Q4)
df_clean['arrival_quarter'] = 'Q' + df_clean['arrival_date'].dt.quarter.astype(str)

# Hierarki Musim (Seasonality)
def assign_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

df_clean['arrival_season'] = df_clean['month_num'].apply(assign_season)
df_clean.drop(columns=['month_num'], inplace=True)

print(f"▶ Fitur 'arrival_date' (datetime) berhasil dibuat.")
print(f"▶ Hierarki Kuartal 'arrival_quarter' berhasil dibuat: {df_clean['arrival_quarter'].value_counts().to_dict()}")
print(f"▶ Hierarki Musim 'arrival_season' berhasil dibuat: {df_clean['arrival_season'].value_counts().to_dict()}")


# =========================================================
# 3. PENYIMPANAN DATA FINAL (hotel_booking_preprocessed.csv)
# =========================================================
print("\n" + "=" * 80)
print("3. PENYIMPANAN DATASET PREPROCESSED FINAL")
print("=" * 80)

output_file = "hotel_booking_preprocessed.csv"
df_clean.to_csv(output_file, index=False)

print(f"▶ Dataset final berhasil disimpan ke '{output_file}'")
print(f"▶ Dimensi Akhir Data (df_clean.shape): {df_clean.shape[0]:,} baris × {df_clean.shape[1]} kolom")
print("\n--- 5 Baris Pertama Data Hasil Preprocessing ---")
display(df_clean.head())

### Evaluasi Semantik & Teoretis Tahap 5 (Data Transformation & Discretization)

#### 1. Analisis Perbandingan Metode Diskretisasi (Slide 26-28)
- **Atribut Target (`lead_time`)**:
  - `lead_time` (waktu tunggu antara tanggal pemesanan hingga tanggal kedatangan) memiliki distribusi yang sangat miring ke kanan (**right-skewed**), di mana mayoritas pemesanan dilakukan secara mendadak (0 s.d. 50 hari), sementara sebagian kecil dilakukan jauh-jauh hari (hingga 700+ hari).
- **Equal-Width Partitioning**:
  - Membagi interval nilai minimum ke maksimum secara matematis merata ($[max - min] / N$).
  - **Kelemahan**: Menghasilkan **distribusi frekuensi yang sangat tidak seimbang**. Bin 1 menampung **75,878 record (87.24% data)**, sementara Bin 4 hanya menampung **26 record (0.03%)**. Hal ini membuat algoritma machine learning gagal menangkap pola pada interval menengah dan panjang.
- **Equal-Frequency (Equal-Depth) Partitioning**:
  - Membagi data berdasarkan kuantil ($N = 4$ kuartil), sehingga setiap kategori menampung porsi data yang seimbang (masing-masing **~25%** data atau ~21,700 record).
  - **Keunggulan**: Sangat adaptif pada data yang miring (*skewed*), menghasilkan pembagian kategori yang proporsional:
    * `Very Short`: 0 - 11 hari
    * `Short`: 12 - 49 hari
    * `Medium`: 50 - 125 hari
    * `Long`: > 125 hari
  - **Kesimpulan**: Kolom `lead_time_category` berbasis *Equal-Frequency* digunakan untuk analisis lebih lanjut.

---

#### 2. Concept Hierarchy Generation & Abstraksi OLAP (Slide 30-32)
- **Konsep "Rolling Up" Dimensi Waktu**:
  - Dalam analisis *Data Warehouse* dan *OLAP (Online Analytical Processing)*, hierarki konsep memungkinkan pemrosesan data dari level rinci (granularitas harian) ke level abstraksi yang lebih tinggi (kuartalan dan musiman).
- **Manfaat Bisnis**:
  - **`arrival_date`** (datetime harian): Berguna untuk pemodelan prediksi deret waktu (*time-series forecasting*) dan deteksi fluktuasi harian.
  - **`arrival_quarter`** (Q1 s.d. Q4): Berguna untuk evaluasi kinerja keuangan dan perencanaan anggaran manajemen hotel per kuartal.
  - **`arrival_season`** (Winter, Spring, Summer, Autumn): Berguna untuk analisis pola musiman (*seasonality pattern*). Misalnya, periode *Summer* (Juni - Agustus) mengalami lonjakan pemesanan liburan keluarga, sehingga manajemen hotel dapat menyesuaikan strategi penetapan harga (*dynamic pricing*).